In [1]:
import json
import os
from pathlib import Path

import pandas as pd

"""
file path
"""
domain = os.path.dirname(os.path.abspath('__file__'))
print(domain)
FILE_PATH_sample = os.path.join(domain, 'datasets', 'tmp_xx_ontology_sample0.xlsx')
FILE_PATH_result = os.path.join(domain, 'output', '20260512_195553')

e:\claude-code\Ontology_AgenticML\Ontology_Marketing_Research


In [2]:
data_sample = pd.read_excel(FILE_PATH_sample)

path = Path(os.path.join(FILE_PATH_result, 'results.json'))
with path.open("r", encoding="utf-8") as f:
    result_dct = json.load(f)

C:\Users\Yuan\AppData\Roaming\Python\Python39\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:
# top1 推荐
client_product_top1 = {}
for i in result_dct:
    for k, v in i.items():
        product_id_lst = []
        for ref_i in v:
            if ref_i.get('rank') == 1:
                product_id_lst.append(ref_i.get("险种代码"))

        client_product_top1.update({k: product_id_lst})

# top3 推荐
client_product_top3 = {}
for i in result_dct:
    for k, v in i.items():
        product_id_lst = []
        for ref_i in v:
            product_id_lst.append(ref_i["险种代码"])

        client_product_top3.update({k: product_id_lst})

In [5]:
client_product_top3

{'803b6215308029e0da0c0211f2587519': ['26621100', '44734500', '27885500'],
 'ac317cfa169e1e03299ec3de553d9de2': ['11430100', '44644700', '44661600'],
 '7654e53b1e704f6e9bf181dd62a56da3': ['11430100', '27885500', '87220200'],
 '0cbc159c4f11fdc0aeea0965680354ca': ['27885500', '27885200', '44819500'],
 '0b8ba6a0594d1626019f6d2b90831d40': ['44644700', '27885500', '73834200']}

In [6]:
df_ground_truth = data_sample[['appnt_id_no','cvrg_cd']].drop_duplicates()

In [7]:
df_ground_truth.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 3625 entries, 0 to 3638
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   appnt_id_no  3625 non-null   object
 1   cvrg_cd      3625 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 85.0+ KB


In [8]:
# top1 推荐结果准确率

records = []
for k, v in client_product_top1.items():
    for code in v:
        records.append((k, code))
check_df = pd.DataFrame(records, columns=['appnt_id_no', 'cvrg_cd'])
check_df['cvrg_cd'] = check_df['cvrg_cd'].astype(int)


merged = check_df.merge(
    df_ground_truth,
    left_on=['appnt_id_no', 'cvrg_cd'],
    right_on=['appnt_id_no', 'cvrg_cd'],
    how='left',
    indicator=True
)

merged['is_hit_top1'] = merged['_merge'] == 'both'

hit_result_top1 = merged[['appnt_id_no', 'cvrg_cd', 'is_hit_top1']]

In [9]:
# top3 推荐结果准确率

records = []
for k, v in client_product_top3.items():
    for code in v:
        records.append((k, code))
check_df = pd.DataFrame(records, columns=['appnt_id_no', 'cvrg_cd'])
check_df['cvrg_cd'] = check_df['cvrg_cd'].astype(int)

merged = check_df.merge(
    df_ground_truth,
    left_on=['appnt_id_no', 'cvrg_cd'],
    right_on=['appnt_id_no', 'cvrg_cd'],
    how='left',
    indicator=True
)

merged['is_hit_top3'] = merged['_merge'] == 'both'

hit_result_top3 = merged[['appnt_id_no', 'cvrg_cd', 'is_hit_top3']]

In [10]:
hit_result_top1_path = os.path.join(FILE_PATH_result, "top1_reference_detail.xlsx")
hit_result_top3_path = os.path.join(FILE_PATH_result, "top3_reference_detail.xlsx")

hit_result_top1.to_excel(hit_result_top1_path, index = False)
hit_result_top3.to_excel(hit_result_top3_path, index = False)


print(f"Top1 产品推荐明细存入 {hit_result_top1_path}")
print(f"Top3 产品推荐明细存入 {hit_result_top3_path}")

Top1 产品推荐明细存入 e:\claude-code\Ontology_AgenticML\Ontology_Marketing_Research\output\20260512_195553\top1_reference_detail.xlsx
Top3 产品推荐明细存入 e:\claude-code\Ontology_AgenticML\Ontology_Marketing_Research\output\20260512_195553\top3_reference_detail.xlsx


In [16]:
top1_precision = hit_result_top1[hit_result_top1.is_hit_top1 == 1].appnt_id_no.nunique() / hit_result_top1.appnt_id_no.nunique()
top3_precision = hit_result_top3[hit_result_top3.is_hit_top3 == 1].appnt_id_no.nunique() / hit_result_top3.appnt_id_no.nunique()

conclusion_top1 = f"""
Top1 产品推荐：
    - 命中数: {hit_result_top1[hit_result_top1.is_hit_top1 == 1].appnt_id_no.nunique()}
    - 总数: {hit_result_top1.appnt_id_no.nunique()}
    - 准确率: {top1_precision:.1%}
"""

conclusion_top3 = f"""
Top3 产品推荐：
    - 命中数: {hit_result_top3[hit_result_top3.is_hit_top3 == 1].appnt_id_no.nunique()}
    - 总数: {hit_result_top3.appnt_id_no.nunique()}
    - 准确率: {top3_precision:.1%}
"""

print(conclusion_top1)
print(conclusion_top3)


Top1 产品推荐：
    - 命中数: 1
    - 总数: 5
    - 准确率: 20.0%


Top3 产品推荐：
    - 命中数: 2
    - 总数: 5
    - 准确率: 40.0%

